<a href="https://colab.research.google.com/github/mdsadaqathali/week6/blob/main/w7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

base_path="ml-100k"
ratings_path=os.path.join(base_path,"u.data")
movies_path=os.path.join(base_path,"u.item")

ratings=pd.read_csv(ratings_path,sep="\t",names=["userId","movieId","rating","timestamp"])

movie_columns=["movieId","title","release_date","video_release_date","IMDb_URL","unknown","Action","Adventure","Animation","Children","Comedy","Crime","Documentary","Drama","Fantasy","Film-Noir","Horror","Musical","Mystery","Romance","Sci-Fi","Thriller","War","Western"]

movies=pd.read_csv(movies_path,sep="|",names=movie_columns,encoding="latin-1")

genre_columns=["unknown","Action","Adventure","Animation","Children","Comedy","Crime","Documentary","Drama","Fantasy","Film-Noir","Horror","Musical","Mystery","Romance","Sci-Fi","Thriller","War","Western"]

movies["genres"]=movies[genre_columns].apply(lambda x:" ".join([g for g,v in zip(genre_columns,x) if v==1]),axis=1)

data=pd.merge(ratings,movies[["movieId","title","genres"]],on="movieId")

print("Dataset Shape:",data.shape)
print("Number of Users:",data["userId"].nunique())
print("Number of Movies:",data["movieId"].nunique())
print("Number of Ratings:",len(data))

print("\nRating Distribution:")
print(data["rating"].value_counts().sort_index())

plt.figure(figsize=(8,5))
sns.countplot(x=data["rating"])
plt.title("Movie Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.show()

user_item=data.pivot_table(index="userId",columns="movieId",values="rating")

print("\nUser-Item Matrix Shape:",user_item.shape)

sparsity=1-(user_item.count().sum()/(user_item.shape[0]*user_item.shape[1]))

print("Sparsity:",round(sparsity,4))

plt.figure(figsize=(12,6))
sns.heatmap(user_item.iloc[:50,:50],cmap="viridis")
plt.title("User-Item Matrix Heatmap")
plt.xlabel("Movie ID")
plt.ylabel("User ID")
plt.show()

tfidf=TfidfVectorizer()
tfidf_matrix=tfidf.fit_transform(movies["genres"])

content_similarity=cosine_similarity(tfidf_matrix)

movie_indices=pd.Series(movies.index,index=movies["title"]).drop_duplicates()

def recommend_similar(movie_title,n=10):
    if movie_title not in movie_indices:
        print("Movie not found.")
        return pd.DataFrame()
    idx=movie_indices[movie_title]
    similarity_scores=list(enumerate(content_similarity[idx]))
    similarity_scores=sorted(similarity_scores,key=lambda x:x[1],reverse=True)
    similarity_scores=similarity_scores[1:n+1]
    movie_index=[x[0] for x in similarity_scores]
    scores=[x[1] for x in similarity_scores]
    return pd.DataFrame({"Movie":movies.iloc[movie_index]["title"].values,"Similarity":scores})

print("\nContent-Based Recommendations:")
print(recommend_similar("Toy Story (1995)",10).to_string(index=False))

user_item_filled=user_item.fillna(0)

user_similarity=cosine_similarity(user_item_filled)

user_similarity_df=pd.DataFrame(user_similarity,index=user_item.index,columns=user_item.index)

def collaborative_recommend(user_id,n=10):
    if user_id not in user_similarity_df.index:
        print("User not found.")
        return pd.DataFrame()
    similar_users=user_similarity_df.loc[user_id].sort_values(ascending=False).iloc[1:6]
    watched=set(user_item.loc[user_id].dropna().index)
    scores={}
    for similar_user,similarity in similar_users.items():
        rated_movies=user_item.loc[similar_user].dropna()
        for movie_id,rating in rated_movies.items():
            if movie_id not in watched and rating>=4:
                scores[movie_id]=scores.get(movie_id,0)+similarity*rating
    recommendations=sorted(scores.items(),key=lambda x:x[1],reverse=True)[:n]
    movie_ids=[x[0] for x in recommendations]
    score_map=dict(recommendations)
    result=movies[movies["movieId"].isin(movie_ids)][["movieId","title"]].copy()
    result["Score"]=result["movieId"].map(score_map)
    return result.sort_values("Score",ascending=False)[["title","Score"]]

print("\nCollaborative Filtering Recommendations:")
print(collaborative_recommend(1,10).to_string(index=False))

reader=Reader(rating_scale=(1,5))

surprise_data=Dataset.load_from_df(ratings[["userId","movieId","rating"]],reader)

factors=[50,100,150,200]
best_rmse=float("inf")
best_factor=50

print("\nSVD Model Tuning:")

for factor in factors:
    model=SVD(n_factors=factor,random_state=42)
    cv_results=cross_validate(model,surprise_data,measures=["RMSE"],cv=5,verbose=False)
    mean_rmse=cv_results["test_rmse"].mean()
    print("n_factors:",factor,"RMSE:",round(mean_rmse,4))
    if mean_rmse<best_rmse:
        best_rmse=mean_rmse
        best_factor=factor

print("Best n_factors:",best_factor)
print("Best Cross-Validation RMSE:",round(best_rmse,4))

svd_model=SVD(n_factors=best_factor,random_state=42)

train_ratings,test_ratings=train_test_split(ratings[["userId","movieId","rating"]],test_size=0.2,random_state=42)

train_data=Dataset.load_from_df(train_ratings,reader)

trainset=train_data.build_full_trainset()

testset=list(test_ratings.itertuples(index=False,name=None))

svd_model.fit(trainset)

predictions=svd_model.test(testset)

rmse=np.sqrt(np.mean([(p.r_ui-p.est)**2 for p in predictions]))

mae=np.mean([abs(p.r_ui-p.est) for p in predictions])

print("\nSVD Test Results:")
print("Test RMSE:",round(rmse,4))
print("Test MAE:",round(mae,4))

def svd_recommend(user_id,n=10):
    if user_id not in ratings["userId"].values:
        print("User not found.")
        return pd.DataFrame()
    watched=set(ratings[ratings["userId"]==user_id]["movieId"])
    candidate_movies=movies[~movies["movieId"].isin(watched)].copy()
    candidate_movies["PredictedRating"]=candidate_movies["movieId"].apply(lambda x:svd_model.predict(user_id,x).est)
    return candidate_movies.sort_values("PredictedRating",ascending=False).head(n)[["title","PredictedRating"]]

print("\nSVD Recommendations:")
print(svd_recommend(1,10).to_string(index=False))

def hybrid_recommend(user_id,movie_title,n=10):
    if user_id not in ratings["userId"].values:
        print("User not found.")
        return pd.DataFrame()
    if movie_title not in movie_indices:
        print("Movie not found.")
        return pd.DataFrame()
    idx=movie_indices[movie_title]
    watched=set(ratings[ratings["userId"]==user_id]["movieId"])
    candidates=movies[~movies["movieId"].isin(watched)].copy()
    candidates["SVDScore"]=candidates["movieId"].apply(lambda x:svd_model.predict(user_id,x).est)
    candidates["ContentScore"]=content_similarity[idx][candidates.index]
    svd_min=candidates["SVDScore"].min()
    svd_max=candidates["SVDScore"].max()
    content_min=candidates["ContentScore"].min()
    content_max=candidates["ContentScore"].max()
    if svd_max>svd_min:
        candidates["SVDNormalized"]=(candidates["SVDScore"]-svd_min)/(svd_max-svd_min)
    else:
        candidates["SVDNormalized"]=0
    if content_max>content_min:
        candidates["ContentNormalized"]=(candidates["ContentScore"]-content_min)/(content_max-content_min)
    else:
        candidates["ContentNormalized"]=0
    candidates["HybridScore"]=0.7*candidates["SVDNormalized"]+0.3*candidates["ContentNormalized"]
    return candidates.sort_values("HybridScore",ascending=False).head(n)[["title","SVDScore","ContentScore","HybridScore"]]

print("\nHybrid Recommendations:")
print(hybrid_recommend(1,"Toy Story (1995)",10).to_string(index=False))

def precision_recall_at_k(predictions,k=10,threshold=4):
    user_est_true={}
    for pred in predictions:
        uid=pred.uid
        true_r=pred.r_ui
        est=pred.est
        user_est_true.setdefault(uid,[]).append((est,true_r))
    precisions=[]
    recalls=[]
    for uid,ratings_list in user_est_true.items():
        ratings_list=sorted(ratings_list,key=lambda x:x[0],reverse=True)
        top_k=ratings_list[:k]
        relevant=sum(true_r>=threshold for _,true_r in ratings_list)
        recommended_relevant=sum(true_r>=threshold for _,true_r in top_k)
        precision=recommended_relevant/len(top_k) if len(top_k)>0 else 0
        recall=recommended_relevant/relevant if relevant>0 else 0
        precisions.append(precision)
        recalls.append(recall)
    return np.mean(precisions),np.mean(recalls)

precision,recall=precision_recall_at_k(predictions,10,4)

print("\nEvaluation:")
print("Precision@10:",round(precision,4))
print("Recall@10:",round(recall,4))

summary=pd.DataFrame({"Approach":["Content-Based","Collaborative Filtering","SVD","Hybrid"],"Technique":["TF-IDF + Cosine Similarity","User-User Cosine Similarity","SVD Matrix Factorization","70% SVD + 30% Content"],"RMSE":[np.nan,np.nan,rmse,np.nan],"Precision@10":[np.nan,np.nan,precision,np.nan],"Recall@10":[np.nan,np.nan,recall,np.nan]})

print("\nFinal Comparison:")
print(summary.to_string(index=False))

print("\nProject Completed Successfully!")